# Benchmark compiled corrected math

Compare native, corrected, and compiled corrected forward/backward operations.
Run in the MLX environment. No datasets or checkpoints are loaded.

`first_ms` includes the first execution and any compilation. `steady_ms` excludes
warmup and synchronizes each result. These are isolated operation timings.

Reference: [MLX compilation](https://ml-explore.github.io/mlx/build/html/usage/compile.html).


In [ ]:
from pathlib import Path
import time
import tomllib
import mlx.core as mx
import mlx.nn as nn
from llm_training.models.numerics_mlx import (
    stable_softmax, stable_cross_entropy,
)

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/llm_training").is_dir())
config_path = root / "configs/pretraining/sequential_gpt_mlx_swa_3to1_stable_compiled.toml"
config = tomllib.loads(config_path.read_text())
batch = config["training"]["batch_size"]
length = config["model"]["context_length"]
heads = config["model"]["n_heads"]
vocab = 8000  # Set to the tokenizer vocabulary used by your run.
iterations = 10
mx.set_default_device(mx.gpu)
mx.random.seed(42)
print(f"Device: {mx.default_device()}; batch={batch}, context={length}, vocab={vocab}")

compiled_stable_softmax = mx.compile(stable_softmax)
compiled_stable_cross_entropy = mx.compile(stable_cross_entropy)


In [ ]:
def benchmark(function, *args):
    mx.eval(*args)
    start = time.perf_counter()
    mx.eval(function(*args))
    first_ms = (time.perf_counter() - start) * 1000
    for _ in range(2):
        mx.eval(function(*args))
    timings = []
    for _ in range(iterations):
        start = time.perf_counter()
        mx.eval(function(*args))
        timings.append((time.perf_counter() - start) * 1000)
    return first_ms, sum(timings) / len(timings)


def report(name, function, *args):
    first, steady = benchmark(function, *args)
    print(f"{name:24} first_ms={first:9.3f} steady_ms={steady:9.3f}")
    return steady


In [ ]:
logits = mx.random.normal((batch, length, vocab))
targets = mx.random.randint(0, vocab, (batch, length))
losses = {
    "native loss": lambda z, t: nn.losses.cross_entropy(z, t, reduction="mean"),
    "corrected loss": stable_cross_entropy,
    "compiled corrected loss": compiled_stable_cross_entropy,
}
loss_times = {}
for name, function in losses.items():
    loss_times[name] = report(name, mx.value_and_grad(function), logits, targets)
print(f"Corrected loss speedup: {loss_times['corrected loss'] / loss_times['compiled corrected loss']:.2f}x")

report("compiled loss + backward", mx.compile(mx.value_and_grad(stable_cross_entropy)), logits, targets)


In [ ]:
# A full-attention softmax row; use the causal mask from training.
positions = mx.arange(length)
mask = positions[:, None] >= positions[None, :]
scores = mx.where(mask, mx.random.normal((batch, heads, length, length)), -float("inf"))
weights = mx.random.normal(scores.shape)
softmaxes = {
    "native softmax": lambda z: mx.softmax(z, axis=-1),
    "corrected softmax": stable_softmax,
    "compiled corrected softmax": compiled_stable_softmax,
}
softmax_times = {}
for name, function in softmaxes.items():
    # Non-constant upstream gradient; summing softmax alone would give zero gradients.
    def objective(z, w, function=function):
        return mx.mean(function(z) * w)
    softmax_times[name] = report(name, mx.value_and_grad(objective), scores, weights)
print(f"Corrected softmax speedup: {softmax_times['corrected softmax'] / softmax_times['compiled corrected softmax']:.2f}x")


In [ ]:
# Full model forward/backward at the config dimensions; no optimizer updates.
from llm_training.models.gpt_mlx import GPT, GPTConfig
from llm_training.pretraining_mlx.train import make_loss_and_grad

settings = dict(config["model"], vocab_size=vocab)
model = GPT(GPTConfig(**settings))
mx.eval(model.parameters())
parameters = model.parameters()
x = mx.random.randint(0, vocab, (batch, length))
y = mx.random.randint(0, vocab, x.shape)
model_times = {}
for name, stable, compiled in (("native model", False, False),
                                ("corrected model", True, False),
                                ("compiled corrected model", True, True)):
    candidate = GPT(GPTConfig(**{**settings, "stable_softmax": stable}))
    candidate.update(parameters)
    mx.eval(candidate.parameters())
    function = make_loss_and_grad(candidate, compiled)
    model_times[name] = report(name, function, x, y)
print(f"Model speedup: {model_times['corrected model'] / model_times['compiled corrected model']:.2f}x")
